<a href="https://colab.research.google.com/github/jacobr12/CS197-ProjectB/blob/main/analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

# Load the dataset
try:
    df = pd.read_csv('analysis_df.csv')
    print("Successfully loaded analysis_df.csv.")
except FileNotFoundError:
    print("Error: 'analysis_df.csv' not found. Please make sure the file is uploaded or the path is correct.")
    # Exit or handle error gracefully
    exit()

# Assuming the first column is 'method' and the second is 'family'
# Adjust these if your CSV headers are different
model_column_name = 'method'
family_column_name = 'family'

# Identify numeric columns (metrics) excluding the model identifiers
# Filtered to only include the metrics explicitly requested by the user
numeric_cols = ['nDCG@k', 'redundancy@k', 'hit_grade3@k', 'semantic_coverage@k', 'sparse_coverage@k']

# Define metrics where 'lower is better'
# Based on user's specific request, only 'redundancy@k' from the kept metrics is 'lower is better'
lower_is_better_metrics = ['redundancy@k']

print("\n--- Analysis of Specific Model Categories ---\n")

# --- Helper function to get top N models for a given family and metric ---
def get_top_n_models(dataframe, family_name, metric, n=3, sort_ascending=False):
    """
    Filters DataFrame by family, sorts by metric, and returns top N rows.
    sort_ascending=True for 'lower is better', False for 'higher is better'.
    """
    filtered_df = dataframe[dataframe[family_column_name] == family_name].copy()
    if not filtered_df.empty:
        # Sort by the specified metric and get the top N
        top_n_df = filtered_df.sort_values(by=metric, ascending=sort_ascending).head(n)
        return top_n_df
    return pd.DataFrame() # Return empty DataFrame if no models found for family


# Store unique models identified and their 'best' metric reason
identified_models = {} # Key: model_name, Value: {'family': family, 'best_metric_reason': metric_name}

# --- Process 'dual_residual' models ---
for metric in numeric_cols:
    sort_asc = metric in lower_is_better_metrics
    top_dual_residual = get_top_n_models(df, 'dual_residual', metric, n=3, sort_ascending=sort_asc)
    if not top_dual_residual.empty:
        for index, row in top_dual_residual.iterrows():
            model_name = row[model_column_name]
            if model_name not in identified_models:
                identified_models[model_name] = {'family': row[family_column_name], 'best_metric_reason': metric}
            # If a model appears multiple times as 'best' for different metrics,
            # we keep the first metric as the reason for its inclusion to avoid repetition.

# --- Process 'fusion' models ---
for metric in numeric_cols:
    sort_asc = metric in lower_is_better_metrics
    top_fusion = get_top_n_models(df, 'fusion', metric, n=3, sort_ascending=sort_asc)
    if not top_fusion.empty:
        for index, row in top_fusion.iterrows():
            model_name = row[model_column_name]
            if model_name not in identified_models:
                identified_models[model_name] = {'family': row[family_column_name], 'best_metric_reason': metric}

# --- Process 'mmr' models ---
for metric in numeric_cols:
    sort_asc = metric in lower_is_better_metrics
    top_mmr = get_top_n_models(df, 'mmr', metric, n=3, sort_ascending=sort_asc)
    if not top_mmr.empty:
        for index, row in top_mmr.iterrows():
            model_name = row[model_column_name]
            if model_name not in identified_models:
                identified_models[model_name] = {'family': row[family_column_name], 'best_metric_reason': metric}

# --- Process 'hybrid_mmr' models ---
for metric in numeric_cols:
    sort_asc = metric in lower_is_better_metrics
    top_hybrid_mmr = get_top_n_models(df, 'hybrid_mmr', metric, n=3, sort_ascending=sort_asc)
    if not top_hybrid_mmr.empty:
        for index, row in top_hybrid_mmr.iterrows():
            model_name = row[model_column_name]
            if model_name not in identified_models:
                identified_models[model_name] = {'family': row[family_column_name], 'best_metric_reason': metric}

# --- Process 'hybrid_pricing' models ---
for metric in numeric_cols:
    sort_asc = metric in lower_is_better_metrics
    top_hybrid_pricing = get_top_n_models(df, 'hybrid_pricing', metric, n=3, sort_ascending=sort_asc)
    if not top_hybrid_pricing.empty:
        for index, row in top_hybrid_pricing.iterrows():
            model_name = row[model_column_name]
            if model_name not in identified_models:
                identified_models[model_name] = {'family': row[family_column_name], 'best_metric_reason': metric}

# --- Process 'dense' model ---
dense_model_df = df[df[model_column_name] == 'dense']
if not dense_model_df.empty:
    model_name = dense_model_df.iloc[0][model_column_name]
    if model_name not in identified_models:
        identified_models[model_name] = {'family': dense_model_df.iloc[0][family_column_name], 'best_metric_reason': "Explicitly requested 'dense' model"}

# Now, build the final DataFrame for display
final_results_list = []
for model_name, details in identified_models.items():
    model_row_data = df[df[model_column_name] == model_name].iloc[0].to_dict()

    # Prepare the dictionary for the final DataFrame row
    display_row = {
        'Method': model_row_data.pop(model_column_name), # Rename and get model name
        'Best Metric (Reason)': details['best_metric_reason'],
        'Family': model_row_data.pop(family_column_name) # Rename and get family
    }

    # Add all numeric metrics to the display row
    for col in numeric_cols:
        display_row[col] = model_row_data.get(col) # Use .get() in case a column isn't found (shouldn't happen here)

    final_results_list.append(display_row)

final_display_df = pd.DataFrame(final_results_list)

# Reorder columns as requested: 'Method', 'Best Metric (Reason)', 'Family', then all numeric metrics
if not final_display_df.empty:
    ordered_columns = ['Method', 'Best Metric (Reason)', 'Family'] + numeric_cols
    # Ensure only existing columns are selected to avoid KeyError if `numeric_cols` was broader than actual data
    final_display_df = final_display_df[[col for col in ordered_columns if col in final_display_df.columns]]

    print("\n### Consolidated Model Analysis Table ###\n")
    print(final_display_df.to_string(index=False))
else:
    print("No models found for analysis based on the criteria.")

print("\n--- Analysis Complete ---")


Successfully loaded analysis_df.csv.

--- Analysis of Specific Model Categories ---


### Consolidated Model Analysis Table ###

                                Method               Best Metric (Reason)         Family   nDCG@k  redundancy@k  hit_grade3@k  semantic_coverage@k  sparse_coverage@k
 dualres_a09_eta005_sd10_b025_r015_s01                             nDCG@k  dual_residual 0.585682      0.694549      0.742268             0.740018           0.940202
dualres_a09_eta005_sd10_b025_r015_s015                             nDCG@k  dual_residual 0.585682      0.694549      0.742268             0.740018           0.940202
 dualres_a09_eta005_sd10_b025_r01_s005                             nDCG@k  dual_residual 0.585682      0.694549      0.742268             0.740018           0.940202
  dualres_a03_eta05_sd20_b075_r005_s01                       redundancy@k  dual_residual 0.304073      0.201103      0.659794             0.691716           0.998527
 dualres_a03_eta05_sd20_b075_r005_s005   

In [2]:
import pandas as pd

# Load the dataset (assuming df is already loaded in cell 5OB0YaN8l_6z)
try:
    # Use the existing df if it's already loaded and valid
    if 'df' not in locals() or df.empty:
        df = pd.read_csv('analysis_df.csv')
        print("Successfully re-loaded analysis_df.csv.")
except FileNotFoundError:
    print("Error: 'analysis_df.csv' not found. Please make sure the file is uploaded or the path is correct.")
    exit()
except Exception as e:
    print(f"An error occurred while loading the DataFrame: {e}")
    exit()

model_column_name = 'method'

# Find the top 10 models by 'redundancy@k' (highest redundancy)
# Assuming a higher value for 'redundancy@k' means 'higher redundancy' as per the user's request.
if 'redundancy@k' in df.columns:
    top_10_redundancy = df.sort_values(by='redundancy@k', ascending=False).head(10)

    print("### Top 10 Configurations with Highest Redundancy@k ###\n")
    print(top_10_redundancy.to_string(index=False))
else:
    print("Error: 'redundancy@k' column not found in the DataFrame.")


### Top 10 Configurations with Highest Redundancy@k ###

                                method         family   nDCG@k  irrelevant@k  redundancy@k  hit_grade3@k  semantic_coverage@k  sparse_coverage@k  docs_retrieved  docs_to_semantic_0.80  docs_to_semantic_0.90  docs_to_semantic_0.95  docs_to_sparse_0.80  docs_to_sparse_0.90  docs_to_sparse_0.95  alpha  lambda_div  eta  sparse_decay  beta_stop  min_combined_residual_frac  min_combined_score
                           fusion_a095         fusion 0.613222      2.216495      0.736243      0.742268             0.739103           0.878497            10.0               5.448276                   4.75               5.666667             2.934211             3.549020             3.387755   0.95         NaN  NaN           NaN        NaN                         NaN                 NaN
                                 dense          dense 0.611521      2.226804      0.735871      0.721649             0.736698           0.870834            10.0   

In [3]:
import pandas as pd

# Load the dataset (assuming df is already loaded from previous cells)
try:
    if 'df' not in locals() or df.empty:
        df = pd.read_csv('analysis_df.csv')
        print("Successfully re-loaded analysis_df.csv.")
except FileNotFoundError:
    print("Error: 'analysis_df.csv' not found. Please make sure the file is uploaded or the path is correct.")
    exit()
except Exception as e:
    print(f"An error occurred while loading the DataFrame: {e}")
    exit()

# Ensure the 'nDCG@k' column exists before attempting to sort
if 'nDCG@k' in df.columns:
    # Sort by 'nDCG@k' in descending order to get the highest values first
    top_10_ndcgs = df.sort_values(by='nDCG@k', ascending=False).head(10)

    print("### Top 10 Configurations with Highest nDCG@k ###\n")
    # Display all columns for the top 10 configurations
    print(top_10_ndcgs.to_string(index=False))
else:
    print("Error: 'nDCG@k' column not found in the DataFrame.")

### Top 10 Configurations with Highest nDCG@k ###

                                method         family   nDCG@k  irrelevant@k  redundancy@k  hit_grade3@k  semantic_coverage@k  sparse_coverage@k  docs_retrieved  docs_to_semantic_0.80  docs_to_semantic_0.90  docs_to_semantic_0.95  docs_to_sparse_0.80  docs_to_sparse_0.90  docs_to_sparse_0.95  alpha  lambda_div  eta  sparse_decay  beta_stop  min_combined_residual_frac  min_combined_score
                           fusion_a095         fusion 0.613222      2.216495      0.736243      0.742268             0.739103           0.878497            10.0               5.448276                   4.75               5.666667             2.934211             3.549020             3.387755   0.95         NaN  NaN           NaN        NaN                         NaN                 NaN
                            fusion_a09         fusion 0.612866      2.288660      0.732885      0.731959             0.732364           0.890721            10.0         